# Bounding box

In [ ]:
import matplotlib.patches as pat
import matplotlib.pyplot as plt
import torch

from unitrack.costs import BoxCIoU, BoxGIoU, BoxIoU
from unitrack.data import Detections, FrameContext, Tracklets
from unitrack.lifecycle import TrackletStatus

boxes1 = torch.tensor([[1.0, 1.0, 12.0, 12.0]])
boxes2 = torch.tensor(
    [
        [3.0, 3.0, 10.0, 10.0],
        [5.0, 5.0, 15.0, 15.0],
        [21.0, 21.0, 30.0, 30.0],
    ]
)

cs = Tracklets(
    id=torch.tensor([0], dtype=torch.int64),
    status=torch.tensor([TrackletStatus.Active], dtype=torch.int8),
    hits=torch.ones(1, dtype=torch.int32),
    time_since_update=torch.zeros(1, dtype=torch.int32),
    age=torch.ones(1, dtype=torch.int32),
    frame_started=torch.zeros(1, dtype=torch.int32),
    frame_last_seen=torch.zeros(1, dtype=torch.int32),
    box=boxes1,
    batch_size=[1],
)
ds = Detections(
    index=torch.arange(3, dtype=torch.int64),
    box=boxes2,
    batch_size=[3],
)
ctx = FrameContext.make(frame_idx=0, delta=1 / 30.0, fps=30.0, stream_key=0)

variants = {
    "IoU": BoxIoU("box"),
    "GIoU": BoxGIoU("box"),
    "CIoU": BoxCIoU("box"),
}

for name, cost in variants.items():
    matrix = cost(cs, ds, ctx).matrix
    fig, ax = plt.subplots()
    ax.set_xbound(0, 50)
    ax.set_ybound(0, 50)
    ax.set_title(f"1 - {name}")
    for x1, y1, x2, y2 in boxes1:
        w = int(x2 - x1)
        h = int(y2 - y1)
        ax.add_patch(pat.Rectangle((x1, y1), w, h, fill=False, edgecolor="r"))
        ax.text(x1, y1, f"{w}x{h}", color="r")
    for i, (x1, y1, x2, y2) in enumerate(boxes2):
        w = int(x2 - x1)
        h = int(y2 - y1)
        ax.add_patch(pat.Rectangle((x1, y1), w, h, fill=False, edgecolor="g"))
        ax.text(x1, y1, f"{matrix[0, i]:.2f}", color="g")


# Embedding vectors

In [ ]:
import matplotlib.pyplot as plt
import torch

from unitrack.costs import BiSoftmax, Cosine
from unitrack.data import Detections, FrameContext, Tracklets
from unitrack.lifecycle import TrackletStatus

vecs1 = torch.tensor([[torch.pi / 4, torch.pi / 4]]) + 2.0
vecs2 = torch.tensor(
    [
        [torch.pi / 3.4, torch.pi / 4.2],
        [torch.pi / 2.1 * 2, torch.pi / 4.1 * 2],
        [0.0, 0.9],
    ]
) + 2.0

cs = Tracklets(
    id=torch.tensor([0], dtype=torch.int64),
    status=torch.tensor([TrackletStatus.Active], dtype=torch.int8),
    hits=torch.ones(1, dtype=torch.int32),
    time_since_update=torch.zeros(1, dtype=torch.int32),
    age=torch.ones(1, dtype=torch.int32),
    frame_started=torch.zeros(1, dtype=torch.int32),
    frame_last_seen=torch.zeros(1, dtype=torch.int32),
    kernel=vecs1,
    batch_size=[1],
)
ds = Detections(
    index=torch.arange(3, dtype=torch.int64),
    kernel=vecs2,
    batch_size=[3],
)
ctx = FrameContext.make(frame_idx=0, delta=1 / 30.0, fps=30.0, stream_key=0)

variants = {
    "1 - cosine": Cosine("kernel"),
    "1 - bisoftmax": BiSoftmax("kernel"),
}

for name, cost in variants.items():
    matrix = cost(cs, ds, ctx).matrix
    fig, ax = plt.subplots()
    ax.set_title(name)
    for vx, vy in vecs1:
        ax.arrow(0, 0, vx, vy, head_width=0.1, head_length=0.1, fc="r", ec="r")
    for i, (vx, vy) in enumerate(vecs2):
        ax.arrow(0, 0, vx, vy, head_width=0.1, head_length=0.1, fc="g", ec="g")
        ax.text(vx + 0.1, vy + 0.1, f"{matrix[0, i]:.2f}", color="g")
